# Phase 1 — Data Collection & Audit (Colab)

**Pre-requisite:** Run `00_colab_setup.ipynb` first.

This notebook:
1. Downloads robot images from Roboflow Universe
2. Helps you collect additional images
3. Loads your annotated dataset exported from Roboflow
4. Runs the mandatory VIZ 1.A / 1.B / 1.C audit

**Target:** ≥300 images, ≥25 instances per class in train set.

In [ ]:
import os, sys, random, collections, shutil
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

PROJECT_DIR = '/content/drive/MyDrive/robot-perception'
DATA_DIR    = f'{PROJECT_DIR}/data/annotated'
RESULTS_DIR = f'{PROJECT_DIR}/results/figures'
os.makedirs(RESULTS_DIR, exist_ok=True)

CLASS_NAMES = ['arm', 'leg', 'torso', 'head', 'sensor']

## Part A — Image Collection via Roboflow Universe

### Step 1: Get your Roboflow API key
1. Go to https://app.roboflow.com
2. Create a free account
3. Go to Settings → API → copy your Private API Key
4. Paste it below

In [ ]:
# ⬇️  FILL IN YOUR API KEY
ROBOFLOW_API_KEY = 'YOUR_API_KEY_HERE'

# Datasets to download from Universe (we take their IMAGES, re-annotate ourselves)
# Format: (workspace_slug, project_slug, version_number)
UNIVERSE_DATASETS = [
    # These datasets have robot images even if class labels differ from ours
    # We download images only and re-annotate with our 5-class taxonomy
    # Add more from: https://universe.roboflow.com/search?q=robot
    ('lab-4ifnn', 'humanoid-robots-and-people-a2inv', 1),
    # Add more datasets you find on Universe:
    # ('workspace', 'project', version),
]

In [ ]:
from roboflow import Roboflow

RAW_DIR = f'{PROJECT_DIR}/data/raw'
os.makedirs(RAW_DIR, exist_ok=True)

rf = Roboflow(api_key=ROBOFLOW_API_KEY)

for workspace, project_slug, version in UNIVERSE_DATASETS:
    print(f'Downloading {workspace}/{project_slug} v{version}...')
    try:
        proj = rf.workspace(workspace).project(project_slug)
        dataset = proj.version(version).download('yolov8', location=f'{RAW_DIR}/{project_slug}')
        print(f'  → saved to {RAW_DIR}/{project_slug}')
    except Exception as e:
        print(f'  ERROR: {e}')

# Count total raw images collected
total_raw = sum(len([f for f in os.listdir(os.path.join(RAW_DIR, d, split))
                      if f.endswith(('.jpg','.png','.jpeg'))])
                 for d in os.listdir(RAW_DIR) if os.path.isdir(os.path.join(RAW_DIR, d))
                 for split in ['train','valid','test']
                 if os.path.exists(os.path.join(RAW_DIR, d, split)))
print(f'\nTotal raw images: {total_raw}')

## Part B — Extract frames from YouTube robot videos

This gives you diverse images of real humanoid robots in motion.
Suggested videos:
- Boston Dynamics Atlas demos (search YouTube for 'Boston Dynamics Atlas 2023')
- Figure AI robot demos  
- 1X Technologies NEO demos
- Agility Robotics Digit

In [ ]:
!pip install -q yt-dlp

# ⬇️  PASTE YOUTUBE URLS HERE (one per line)
YOUTUBE_URLS = [
    # 'https://www.youtube.com/watch?v=EXAMPLE1',
    # 'https://www.youtube.com/watch?v=EXAMPLE2',
]

VIDEO_DIR = f'{PROJECT_DIR}/data/raw/videos'
FRAMES_DIR = f'{PROJECT_DIR}/data/raw/video_frames'
os.makedirs(VIDEO_DIR, exist_ok=True)
os.makedirs(FRAMES_DIR, exist_ok=True)

for url in YOUTUBE_URLS:
    video_id = url.split('v=')[-1][:11]
    out_path = f'{VIDEO_DIR}/{video_id}.mp4'
    if not os.path.exists(out_path):
        !yt-dlp -f 'best[height<=720]' -o '{out_path}' '{url}'

    # Extract frames every 30 frames (~1 per second at 30fps)
    frame_out = f'{FRAMES_DIR}/{video_id}'
    os.makedirs(frame_out, exist_ok=True)
    cap = cv2.VideoCapture(out_path)
    idx, saved = 0, 0
    while True:
        ret, frame = cap.read()
        if not ret: break
        if idx % 30 == 0:
            cv2.imwrite(f'{frame_out}/frame_{idx:06d}.jpg', frame)
            saved += 1
        idx += 1
    cap.release()
    print(f'{video_id}: extracted {saved} frames')

## Part C — Load YOUR annotated dataset from Roboflow

After collecting and annotating images in Roboflow:
1. Go to your Roboflow project → Versions → Generate a version
2. Export format: **YOLOv8**
3. Copy the download code snippet and paste below

This downloads your annotated dataset directly into the Drive folder.

In [ ]:
# ⬇️  FILL IN YOUR project details after annotation is done
MY_WORKSPACE  = 'your-workspace-name'
MY_PROJECT    = 'robot-parts'   # whatever you named your project
MY_VERSION    = 1

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
dataset = rf.workspace(MY_WORKSPACE).project(MY_PROJECT).version(MY_VERSION).download(
    'yolov8',
    location=f'{PROJECT_DIR}/data/roboflow_export'
)
print('Dataset downloaded.')

In [ ]:
# Reorganize Roboflow export into our folder structure
# Roboflow exports as: train/images, train/labels, valid/images, valid/labels, test/...
# We need: images/train, labels/train, etc.

EXPORT_DIR = f'{PROJECT_DIR}/data/roboflow_export'

SPLIT_MAP = {'train': 'train', 'valid': 'val', 'test': 'test'}

for rf_split, our_split in SPLIT_MAP.items():
    for kind in ('images', 'labels'):
        src = os.path.join(EXPORT_DIR, rf_split, kind if kind == 'images' else 'labels')
        dst = os.path.join(DATA_DIR, kind, our_split)
        if not os.path.exists(src):
            # Roboflow sometimes uses 'labels' subdir name
            alt = os.path.join(EXPORT_DIR, rf_split, 'labels')
            src = alt if kind == 'labels' else src
        if os.path.exists(src):
            os.makedirs(dst, exist_ok=True)
            for f in os.listdir(src):
                shutil.copy2(os.path.join(src, f), os.path.join(dst, f))
            print(f'  {rf_split}/{kind} → {kind}/{our_split} ({len(os.listdir(dst))} files)')

print('\nReorganization done.')

In [ ]:
# Carve out calibration set (100 images from train) — needed for Phase 6 conformal prediction
# Run this ONCE, then never touch the calibration split again

import shutil, random

CALIB_N = 100
train_imgs = [f for f in os.listdir(f'{DATA_DIR}/images/train')
              if f.endswith(('.jpg','.jpeg','.png'))]

existing_calib = [f for f in os.listdir(f'{DATA_DIR}/images/calibration')
                  if f.endswith(('.jpg','.jpeg','.png'))]

if existing_calib:
    print(f'Calibration set already exists ({len(existing_calib)} images). Skipping.')
elif len(train_imgs) < CALIB_N + 50:  # need at least 50 left for train
    print(f'Not enough train images ({len(train_imgs)}) to carve out {CALIB_N}. Collect more first.')
else:
    random.seed(42)
    calib_imgs = random.sample(train_imgs, CALIB_N)
    for fname in calib_imgs:
        stem = os.path.splitext(fname)[0]
        # Move image
        shutil.move(f'{DATA_DIR}/images/train/{fname}',
                    f'{DATA_DIR}/images/calibration/{fname}')
        # Move label
        lbl = f'{DATA_DIR}/labels/train/{stem}.txt'
        if os.path.exists(lbl):
            shutil.move(lbl, f'{DATA_DIR}/labels/calibration/{stem}.txt')
    print(f'Moved {CALIB_N} images to calibration set.')
    print(f'Train set remaining: {len(train_imgs) - CALIB_N} images')

## VIZ 1.A — Class distribution bar chart

In [ ]:
def count_class_distribution(labels_dir):
    counts = collections.Counter()
    if not os.path.exists(labels_dir):
        return counts
    for label_file in os.listdir(labels_dir):
        if not label_file.endswith('.txt'): continue
        with open(os.path.join(labels_dir, label_file)) as f:
            for line in f:
                line = line.strip()
                if line:
                    counts[int(line.split()[0])] += 1
    return counts

train_counts = count_class_distribution(f'{DATA_DIR}/labels/train')
bar_values = [train_counts[i] for i in range(5)]

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(CLASS_NAMES, bar_values, color='steelblue', edgecolor='white')
ax.axhline(y=25, color='red', linestyle='--', label='min threshold (25)')
for bar, count in zip(bars, bar_values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            str(count), ha='center', va='bottom', fontsize=11)
ax.set_ylabel('Annotated instances (train)')
ax.set_title('VIZ 1.A — Class distribution (training set)')
ax.legend()
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/viz1a_class_distribution.png', dpi=150)
plt.show()
print('⚠️  Any bar below red line = not enough data. Fix before training.')

## VIZ 1.B — 20 random annotated images

In [ ]:
def draw_annotations(img_path, label_path):
    img = cv2.imread(img_path)
    if img is None: return np.zeros((224,224,3), dtype=np.uint8)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    COLORS_BGR = [(255,80,80),(80,200,80),(80,80,255),(255,200,0),(200,80,255)]
    if os.path.exists(label_path):
        with open(label_path) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) < 5: continue
                cls = int(parts[0])
                cx,cy,bw,bh = float(parts[1]),float(parts[2]),float(parts[3]),float(parts[4])
                x1=int((cx-bw/2)*w); y1=int((cy-bh/2)*h)
                x2=int((cx+bw/2)*w); y2=int((cy+bh/2)*h)
                color = COLORS_BGR[cls % 5]
                cv2.rectangle(img,(x1,y1),(x2,y2),color,2)
                cv2.putText(img,CLASS_NAMES[cls],(x1,max(y1-6,0)),
                            cv2.FONT_HERSHEY_SIMPLEX,0.55,color,2)
    return img

img_dir = f'{DATA_DIR}/images/train'
lbl_dir = f'{DATA_DIR}/labels/train'
img_files = [f for f in os.listdir(img_dir) if f.lower().endswith(('.jpg','.jpeg','.png'))]
sample = random.sample(img_files, min(20, len(img_files)))

fig, axes = plt.subplots(4, 5, figsize=(20, 16))
for ax, fname in zip(axes.flatten(), sample):
    stem = os.path.splitext(fname)[0]
    img = draw_annotations(f'{img_dir}/{fname}', f'{lbl_dir}/{stem}.txt')
    ax.imshow(img); ax.set_title(fname[:20], fontsize=8); ax.axis('off')
for ax in axes.flatten()[len(sample):]: ax.axis('off')
plt.suptitle('VIZ 1.B — 20 random annotated training images', fontsize=14)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/viz1b_annotation_sample.png', dpi=120)
plt.show()
print('⚠️  MANUAL CHECK: tight boxes? correct class labels? No whole-image boxes?')

## VIZ 1.C — Bounding box size distribution

In [ ]:
widths, heights = [], []
for lf in os.listdir(f'{DATA_DIR}/labels/train'):
    if not lf.endswith('.txt'): continue
    with open(f'{DATA_DIR}/labels/train/{lf}') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 5:
                widths.append(float(parts[3]))
                heights.append(float(parts[4]))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(widths, bins=30, color='steelblue', edgecolor='white')
axes[0].axvline(x=0.8, color='red', linestyle='--', label='>0.8 suspicious')
axes[0].set_xlabel('Normalized box width'); axes[0].set_title('Box width distribution')
axes[0].legend()
axes[1].hist(heights, bins=30, color='darkorange', edgecolor='white')
axes[1].axvline(x=0.8, color='red', linestyle='--', label='>0.8 suspicious')
axes[1].set_xlabel('Normalized box height'); axes[1].set_title('Box height distribution')
axes[1].legend()
plt.suptitle('VIZ 1.C — Bounding box size distribution')
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/viz1c_box_sizes.png', dpi=150)
plt.show()

print(f'Total annotations: {len(widths)}')
print(f'Boxes width  > 0.8: {sum(w>0.8 for w in widths)}  ← annotation errors if high')
print(f'Boxes height > 0.8: {sum(h>0.8 for h in heights)}  ← annotation errors if high')
print(f'Boxes width  < 0.02: {sum(w<0.02 for w in widths)}  ← too small to be useful')

## Phase 1 Completion Checklist

Before moving to Phase 2, verify:

- [ ] ≥ 300 total images (train + val + test)
- [ ] All images annotated in YOLO format
- [ ] Train/val/test/calibration split done
- [ ] Calibration set: exactly 100 images carved from train
- [ ] **VIZ 1.A saved** — all class bars ≥ 25
- [ ] **VIZ 1.B saved** — boxes visually correct
- [ ] **VIZ 1.C saved** — no mass of boxes > 0.8
- [ ] 100% annotation coverage (every image has a label file)

If all checked, open `02_baseline_train.ipynb`.